# Exploring the AIID GraphQL API

Poke at the schema and the data **before** committing to a full fetch.
This notebook imports `build/fetch_aiid.py` as a library, so the client,
introspection and selection builder behave exactly as the CLI does.

Run from the repo root. Needs `pandas`; the token is read from
`~/.config/aiid/token` as usual.

Nothing here writes to `data/`. Anything saved lands in `resources/aiid/`.


In [1]:
import importlib.util, json, os, re, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'build' / 'fetch_aiid.py').exists():
    ROOT = ROOT.parent          # started inside notebooks/
assert (ROOT / 'build' / 'fetch_aiid.py').exists(), f'run from the repo root, not {Path.cwd()}'

spec = importlib.util.spec_from_file_location('fa', ROOT / 'build' / 'fetch_aiid.py')
fa = importlib.util.module_from_spec(spec); spec.loader.exec_module(fa)

token, where = fa.resolve_token(None)
print('token source:', where)
client = fa.Client(token)

def run(query, **variables):
    """Ad-hoc GraphQL. Raises on errors instead of returning a half-result."""
    return client.query(query, variables)


token source: /home/nfcom/.config/aiid/token


## 1. What collections exist?

One introspection call gives the whole schema. Everything below reuses it,
so this is the only schema request you pay for.


In [2]:
schema = fa.Schema(client.query(fa.INTROSPECT))

roots = []
for name, field in schema.query_fields.items():
    t = fa.named_type(field['type']).get('name')
    roots.append({'query': name, 'returns': t})
pd.DataFrame(roots).sort_values('query').reset_index(drop=True)


,query,returns
0,ObjectId,ObjectId
1,apiUsage,ApiUsage
2,apiUsageSummaries,ApiUsageSummary
3,apiUsages,ApiUsage
4,candidate,Candidate
5,candidates,Candidate
6,checklist,Checklist
7,checklists,Checklist
8,classification,Classification
9,classifications,Classification


## 2. What does one record look like?

The field tree, with the markers the CLI uses: fields needing arguments,
fields past the depth limit, and cycles.


In [3]:
fa.print_fields(schema, schema.root_type('incidents'), depth=2)


  scalars: _id, date, description, editor_notes, date_modified, incident_id, title, editor_dissimilar_incidents, editor_similar_incidents, flagged_dissimilar_incidents, created_at
  AllegedDeployerOfAISystem: Entity
      scalars: _id, entity_id, name, created_at, date_modified
  AllegedDeveloperOfAISystem: Entity
      scalars: _id, entity_id, name, created_at, date_modified
  AllegedHarmedOrNearlyHarmedParties: Entity
      scalars: _id, entity_id, name, created_at, date_modified
  classifications: Classification
      scalars: _id, namespace, notes, publish
      attributes: Attribute
          scalars: short_name, value_json
      incidents: Incident  [cyclic - not fetched]
      reports: Report
          scalars: _id, authors, cloudinary_id, created_at, date_downloaded, date_modified, date_published, date_submitted, description, editor_notes, epoch_date_published, epoch_date_submitted, flag, image_url, inputs_outputs, is_incident_report, language, plain_text, report_number, source

## 3. Pull one record and look at it

`--depth 2` on a single incident shows the shape of every nested object.


In [4]:
builder = fa.SelectionBuilder(schema, depth=2, exclude=['embedding', 'tsne'])
selection = builder.build(schema.root_type('incidents'))
builder.report()

one = run(fa.page_query('incidents', selection, True, 'incident_id'), limit=1, skip=0)['incidents'][0]
print(json.dumps(one, indent=2, ensure_ascii=False)[:3000])


  needs arguments, skipped: classifications.reports.translations, reports.translations, translations
  beyond --depth, skipped: classifications.reports.user, reports.user.adminData
  cyclic, skipped: classifications.incidents (Incident)


{
  "_id": "625763db343edc875fe639ff",
  "date": "2015-05-19",
  "description": "YouTube’s content filtering and recommendation algorithms exposed children to disturbing and inappropriate videos.",
  "editor_notes": "",
  "date_modified": null,
  "incident_id": 1,
  "title": "Google’s YouTube Kids App Presents Inappropriate Content",
  "AllegedDeployerOfAISystem": [
    {
      "_id": "636f26b6343758cf81522afb",
      "entity_id": "youtube",
      "name": "YouTube",
      "created_at": null,
      "date_modified": null
    }
  ],
  "AllegedDeveloperOfAISystem": [
    {
      "_id": "636f26b6343758cf81522afb",
      "entity_id": "youtube",
      "name": "YouTube",
      "created_at": null,
      "date_modified": null
    }
  ],
  "AllegedHarmedOrNearlyHarmedParties": [
    {
      "_id": "64180be0d84ce592b95028a3",
      "entity_id": "minors",
      "name": "Minors",
      "created_at": null,
      "date_modified": "2026-05-28T13:27:38.743Z"
    }
  ],
  "classifications": [
    {
     

## 4. How big is a record? (choose a page size)

The API refuses responses over 6 MB. Measure before a long run: the script
will shrink its page size automatically, but each shrink costs a round trip.


In [5]:
CAP = 6_291_556  # the server's limit, from the 502 body

def weigh(depth, exclude=(), n=5):
    b = fa.SelectionBuilder(schema, depth, list(exclude))
    sel = b.build(schema.root_type('incidents'))
    rows = run(fa.page_query('incidents', sel, True, 'incident_id'), limit=n, skip=0)['incidents']
    per = len(json.dumps(rows).encode()) / len(rows)
    return {'depth': depth, 'exclude': ','.join(exclude) or '-',
            'kb_per_record': round(per / 1024, 1), 'safe_page': max(1, int(CAP * 0.8 // per))}

pd.DataFrame([
    weigh(0),
    weigh(1, ['reports', 'embedding', 'tsne']),
    weigh(1, ['embedding', 'tsne']),
    weigh(1),
])


,depth,exclude,kb_per_record,safe_page
0,0,-,0.5,10390
1,1,"reports,embedding,tsne",2.3,2159
2,1,"embedding,tsne",199.4,24
3,1,-,216.0,22


## 5. Sample some records into a DataFrame

`--max-records` equivalent. Keep it small while exploring.


In [6]:
SAMPLE = 200
EXCLUDE = ['reports', 'embedding', 'tsne', 'editors']

b = fa.SelectionBuilder(schema, depth=1, exclude=EXCLUDE)
sel = b.build(schema.root_type('incidents'))
rows = fa.fetch_all(client, 'incidents', sel, limit=50, sort_field='incident_id', max_records=SAMPLE)

df = pd.DataFrame([{k: fa.flatten(v) for k, v in r.items()} for r in rows])
print(df.shape)
df.head()


  fetched 100 incidents


(200, 17)


  fetched 200 incidents


,_id,date,description,editor_notes,date_modified,incident_id,title,AllegedDeployerOfAISystem,AllegedDeveloperOfAISystem,AllegedHarmedOrNearlyHarmedParties,classifications,implicated_systems,editor_dissimilar_incidents,editor_similar_incidents,flagged_dissimilar_incidents,nlp_similar_incidents,created_at
0,625763db343edc875fe639ff,2015-05-19,YouTube’s content filtering and recommendation...,,,1,Google’s YouTube Kids App Presents Inappropria...,"[{""_id"": ""636f26b6343758cf81522afb"", ""entity_i...","[{""_id"": ""636f26b6343758cf81522afb"", ""entity_i...","[{""_id"": ""64180be0d84ce592b95028a3"", ""entity_i...","[{""_id"": ""614d65aa049cf2ce54aa3af5"", ""namespac...",,808,,55,"[{""incident_id"": 55, ""similarity"": 0.999094188...",2015-05-19T00:00:00.000Z
1,625763dc343edc875fe63a00,2018-12-05,Twenty-four Amazon workers in New Jersey were ...,,,2,Warehouse robot ruptures can of bear spray and...,"[{""_id"": ""636f26b7343758cf81522afd"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522afd"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522afe"", ""entity_i...","[{""_id"": ""60dd465f80935bc89e6f9b00"", ""namespac...",,,,,"[{""incident_id"": 24, ""similarity"": 0.999084353...",2018-12-05T00:00:00.000Z
2,625763dc343edc875fe63a01,2018-10-27,"A Boeing 737 crashed into the sea, killing 189...",,,3,Crashes with Maneuvering Characteristics Augme...,"[{""_id"": ""636f26b7343758cf81522aff"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522aff"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522b00"", ""entity_i...","[{""_id"": ""60dd465f80935bc89e6f9b01"", ""namespac...",,,,,"[{""incident_id"": 20, ""similarity"": 0.998835861...",2018-10-27T00:00:00.000Z
3,625763dc343edc875fe63a02,2018-03-18,An Uber autonomous vehicle (AV) in autonomous ...,,,4,Uber AV Killed Pedestrian in Arizona,"[{""_id"": ""636f26b7343758cf81522b02"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522b02"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522b04"", ""entity_i...","[{""_id"": ""60dd465f80935bc89e6f9b02"", ""namespac...",,302,297,,"[{""incident_id"": 20, ""similarity"": 0.999603569...",2018-03-18T00:00:00.000Z
4,625763dc343edc875fe63a03,2015-07-13,Study on database reports of robotic surgery m...,,,5,Collection of Robotic Surgery Malfunctions,"[{""_id"": ""636f26b7343758cf81522b05"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522b07"", ""entity_i...","[{""_id"": ""636f26b7343758cf81522b08"", ""entity_i...","[{""_id"": ""60dd465f80935bc89e6f9b03"", ""namespac...",,,,,"[{""incident_id"": 40, ""similarity"": 0.998758375...",2015-07-13T00:00:00.000Z


## 6. Which fields are actually populated?

Decides what is worth fetching. A field empty in 95% of rows is not worth the bytes.


In [7]:
fill = (df.replace('', pd.NA).notna().mean() * 100).round(1).sort_values(ascending=False)
fill.to_frame('percent_filled')


,percent_filled
_id,100.0
date,100.0
description,100.0
title,100.0
incident_id,100.0
AllegedDeployerOfAISystem,100.0
AllegedDeveloperOfAISystem,100.0
classifications,100.0
AllegedHarmedOrNearlyHarmedParties,100.0
created_at,100.0


## 7. Shape of the data


In [8]:
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
display(df['year'].value_counts().sort_index().to_frame('incidents'))

deployers = df['AllegedDeployerOfAISystem'].str.split('; ').explode().str.strip()
display(deployers[deployers.ne('')].value_counts().head(15).to_frame('incidents'))


,incidents
year,
1983,1
1996,1
1998,1
1999,1
2003,1
2007,1
2008,2
2010,1
2011,2


,incidents
AllegedDeployerOfAISystem,
"[{""_id"": ""636f26b8343758cf81522b19"", ""entity_id"": ""google"", ""name"": ""Google"", ""created_at"": null, ""date_modified"": null}]",14
"[{""_id"": ""636f26b7343758cf81522afd"", ""entity_id"": ""amazon"", ""name"": ""Amazon"", ""created_at"": null, ""date_modified"": null}]",13
"[{""_id"": ""636f26b8343758cf81522b1f"", ""entity_id"": ""tesla"", ""name"": ""Tesla"", ""created_at"": null, ""date_modified"": null}]",10
"[{""_id"": ""636f26bb343758cf81522b70"", ""entity_id"": ""facebook"", ""name"": ""Facebook"", ""created_at"": null, ""date_modified"": null}]",10
"[{""_id"": ""636f26b8343758cf81522b30"", ""entity_id"": ""unknown"", ""name"": ""Unknown"", ""created_at"": null, ""date_modified"": ""2026-05-28T11:52:14.624Z""}]",4
"[{""_id"": ""636f26ba343758cf81522b64"", ""entity_id"": ""openai"", ""name"": ""OpenAI"", ""created_at"": null, ""date_modified"": null}]",4
"[{""_id"": ""636f26bf343758cf81522bf6"", ""entity_id"": ""tiktok"", ""name"": ""TikTok"", ""created_at"": null, ""date_modified"": null}]",4
"[{""_id"": ""636f26b6343758cf81522afb"", ""entity_id"": ""youtube"", ""name"": ""YouTube"", ""created_at"": null, ""date_modified"": null}]",3
"[{""_id"": ""636f26c4343758cf81522c76"", ""entity_id"": ""cruise"", ""name"": ""Cruise"", ""created_at"": null, ""date_modified"": null}]",3


## 8. Prefilter for multi-agent candidates

The inclusion test in `docs/methodology.md`: the behaviour must arise from
**interaction between two or more AI agents**. Most of AIID is single-agent,
so narrow the reading list before screening by hand.

This is a recall-first keyword net, not a classifier. It produces a shortlist
to read, and every hit still needs verifying against the primary source.


In [9]:
TERMS = [
    r'multi-?agent', r'agent[- ]to[- ]agent', r'\bagents\b',
    r'collusion|collude|colluding', r'coordinat(?:e|ion|ing)',
    r'negotiat(?:e|ion|ing)', r'bot[- ]to[- ]bot', r'swarm',
    r'prompt injection', r'trading algorithm|algorithmic trading|flash crash',
    r'price[- ]fixing|pricing algorithm',
]
# Non-capturing throughout: capture groups would break str.findall and warn in pandas.
PATTERN = re.compile('|'.join(f'(?:{t})' for t in TERMS), re.I)

text = (df['title'].fillna('') + ' ' + df['description'].fillna(''))
hits = df[text.str.contains(PATTERN)].copy()
hits['why'] = text[hits.index].str.findall(PATTERN).str.join('; ')
print(f'{len(hits)} of {len(df)} sampled incidents match')
hits[['incident_id', 'date', 'title', 'why']]


3 of 200 sampled incidents match


,incident_id,date,title,why
27,28,2010-05-08,2010 Market Flash Crash,Flash Crash
64,65,2016-12-22,Reinforcement Learning Reward Functions in Vid...,agents
87,88,2017-08-15,"""Jewish Baby Strollers"" Provided Anti-Semitic ...",coordinate


## 9. Read the full text of one candidate

Reports hold the narrative. Fetch them for specific incidents rather than
dragging every report through the incidents query.


In [12]:
INCIDENT_ID = int(hits['incident_id'].iloc[1]) if len(hits) else 1

q = '''
query OneIncident($id: Int!) {
  incidents(filter: { incident_id: { EQ: $id } }) {
    incident_id title description
    reports { report_number title url source_domain date_published text }
  }
}'''
try:
    inc = run(q, id=INCIDENT_ID)['incidents'][0]
    print(inc['title'], '\\n')
    for r in inc['reports'][:3]:
        print('-', r['source_domain'], r['date_published'], r['title'])
        print(' ', (r['text'] or '')[:400].replace('\\n', ' '), '...\\n')
except fa.GraphQLError as exc:
    print('filter syntax differs on this schema:', exc)
    print("inspect the IncidentFilter input type for the right syntax")


Reinforcement Learning Reward Functions in Video Games \n
- blog.openai.com 2016-12-22T00:00:00.000Z Faulty Reward Functions in the Wild
  At OpenAI, we've recently started using Universe, our software for measuring and training AI agents, to conduct new RL experiments. Sometimes these experiments illustrate some of the issues with RL as currently practiced. In the following example we'll highlight what happens when a misspecified reward function encourages an RL agent to subvert its environment by prioritizing the acquisition of rew ...\n


## 10. Turn the exploration into a CLI command

Once the selection is right, print it and run the fetch from the shell —
a long download does not belong in a notebook kernel.


In [ ]:
b = fa.SelectionBuilder(schema, depth=1, exclude=EXCLUDE)
print(fa.page_query('incidents', b.build(schema.root_type('incidents')), True, 'incident_id'))
print('\\nEquivalent command:')
print(f"python3 build/fetch_aiid.py --depth 1 --exclude {','.join(EXCLUDE)}")


## 11. Save the shortlist

To `resources/`, never to `data/`. Promoting a row into `data/incidents.csv`
is a manual step after verifying it against the primary source.


In [ ]:
out = ROOT / 'resources' / 'aiid'
out.mkdir(parents=True, exist_ok=True)
path = out / 'candidates_multiagent.csv'
hits.to_csv(path, index=False)
print('wrote', path, len(hits), 'rows')
